# 2. FNO · 입력장에서 해를 예측하기

주기적 단위 정사각형 $[0,1)^2$에서 다음 문제를 다룹니다.

$$u-\Delta u=f.$$

입력은 $64\times64$ forcing field $f$, 출력은 같은 크기의 solution field $u$입니다. FNO는 여러 $(f,u)$ 쌍으로 이 관계를 학습합니다. 앞의 PINN과는 다른 방정식입니다.

**AI에 두 함수를 요청하고 직접 붙여 넣은 뒤, 준비된 PyTorch 학습과 평가 코드를 실행합니다.**


## 1. 준비


In [ ]:
from pathlib import Path
import sys
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/pinn_basics.py").is_file()
             and (p / "02_FNO.ipynb").is_file()), None)
if ROOT is None:
    raise RuntimeError("워크숍 교재 폴더 안에서 이 노트북을 여세요.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("교재:", ROOT)
print("Python:", sys.executable)


In [ ]:
import json
from uuid import uuid4
import numpy as np
import torch
from torch.utils.data import TensorDataset
from physicsnemo.models.fno import FNO
from src.operator_training import run
from IPython.display import Image, display

if not torch.cuda.is_available():
    raise RuntimeError("00_Start_Here의 GPU 확인부터 진행하세요.")
DEVICE = "cuda"
DATA_DIR = ROOT / "data/reaction_diffusion"
CONFIG = ROOT / "config/config_FNO.yaml"
if not all((DATA_DIR / f"{split}.hdf5").exists() for split in ("train", "val", "test")):
    raise RuntimeError("00_Start_Here에서 데이터를 준비하세요.")
FNO_COMPLETED = False


## 2. 두 함수가 맡은 일

`build_datasets`는 준비된 train, validation, test tensor 쌍을 각각 `TensorDataset`으로 감쌉니다. 세 split을 섞거나 다시 정규화하지 않습니다.

`build_model`은 입력과 출력 channel이 1인 2차원 PhysicsNeMo FNO를 만듭니다. 모델 설정은 `model_config`로 이미 전달됩니다.

아래 요청과 코드 틀을 함께 Gemini 또는 ChatGPT에 복사하세요.

> PhysicsNeMo 2.2.2 실습 중이야. 아래 두 함수의 본문만 완성해 줘. `TensorDataset`과 `FNO`는 이미 import되어 있어. 첫 함수는 `train_pairs`, `val_pairs`, `test_pairs`를 각각 `TensorDataset(*pairs)`로 감싸 같은 순서의 3개 tuple을 반환해야 해. split이나 tensor 값을 바꾸면 안 돼. 두 번째 함수는 `FNO(in_channels=1, out_channels=1, dimension=2, **model_config)`를 반환해야 해. `grid_size`는 함수 형식을 유지하고, 새 모델 설정이나 학습 코드를 추가하지 마. 두 함수의 Python 코드만 출력해 줘.

```python
def build_datasets(train_pairs, val_pairs, test_pairs):
    # 세 독립 split을 각각 TensorDataset으로 감싸 반환
    pass

def build_model(model_config, grid_size):
    # [batch, 1, 64, 64] → [batch, 1, 64, 64]의 FNO 생성
    pass
```


In [ ]:
# AI가 작성한 두 함수로 아래 빈 구현을 교체하고 이 셀을 실행하세요.
def build_datasets(train_pairs, val_pairs, test_pairs):
    raise NotImplementedError("AI와 함께 build_datasets를 완성하세요.")

def build_model(model_config, grid_size):
    raise NotImplementedError("AI와 함께 build_model을 완성하세요.")


## 3. 2-step 실행 확인

완성한 함수의 자료형과 shape, split 보존을 기존 코드가 확인합니다. 이어서 2 step만 실행합니다. 이 단계는 실행 확인이며 정확도 검증이 아닙니다.


In [ ]:
SMOKE_RUN = ROOT / "runs" / f"fno-check-{uuid4().hex[:10]}"
smoke_metrics = run(1, build_model, dataset_builder=build_datasets, argv=[
    "--config", str(CONFIG), "--data-dir", str(DATA_DIR), "--device", DEVICE,
    "--steps", "2", "--seed", "42", "--output-dir", str(SMOKE_RUN)])
print("실행 경로 확인 완료. 다음 셀에서 본 학습을 진행합니다.")


## 4. 3,000-step 본 학습

원본과 같은 seed 42, batch 32, learning rate 0.001, 3,000회 Adam update를 사용합니다. FNO Level 1의 loss는 데이터 오차입니다. PDE 잔차는 평가에 사용하며 loss에 더하지 않습니다.


In [ ]:
FNO_COMPLETED = False
FNO_RUN = ROOT / "runs" / f"fno-{uuid4().hex[:10]}"
fno_metrics = run(1, build_model, dataset_builder=build_datasets, argv=[
    "--config", str(CONFIG), "--data-dir", str(DATA_DIR), "--device", DEVICE,
    "--steps", "3000", "--seed", "42", "--output-dir", str(FNO_RUN)])
FNO_COMPLETED = True
print("결과:", FNO_RUN.relative_to(ROOT))


## 5. 실제 test 결과 확인


In [ ]:
if not FNO_COMPLETED:
    raise RuntimeError("3,000-step 본 학습 셀을 완료하세요.")
saved_metrics = json.loads((FNO_RUN / "metrics.json").read_text())
print("Test relative L2:", saved_metrics["test"]["relative_l2"])
print("Test RMSE:", saved_metrics["test"]["rmse"])
print("Test PDE residual RMSE:", saved_metrics["test"]["pde_rmse_fft"])
print("Training seconds:", saved_metrics["training_seconds"])
print("시간 범위: optimizer loop, batch transfer와 logging; 준비/평가/저장은 제외")
display(Image(filename=str(FNO_RUN / "preview.png")))


그래프는 test sample 하나의 예시입니다. 위 수치는 test sample 1,000개 전체를 평가한 값입니다.

## 6. 강사와 함께 비교

| 질문 | PINN | FNO Level 1 |
| --- | --- | --- |
| 신경망 입력 | 좌표 x | 전체 forcing field f |
| 출력 | 해당 좌표의 u | 전체 solution field u |
| 학습 loss | 방정식 잔차와 경계조건 | 예측장과 정답장의 데이터 오차 |
| 정답 데이터 사용 | 해석해는 평가에만 사용 | train의 (f,u) 쌍으로 학습 |

FNO 예측이 그림에서 비슷해도 PDE 잔차를 별도로 확인하는 이유는 무엇인가요? 학습에 쓰지 않은 test field를 나눠 둔 이유는 무엇인가요?

## 선택: 함수 작성이 막힌 경우

강사 안내에 따라 아래 import의 주석을 풀고 실행한 뒤 **3번 실행 확인부터** 다시 진행할 수 있습니다. 같은 학습과 평가를 사용하며 자동으로 정답을 선택하지 않습니다.


In [ ]:
# from instructor.fno_reference import build_datasets, build_model
